# Phân loại ảnh ALL — huấn luyện và so sánh mô hình trên Google Colab

**Chuẩn bị (làm 1 lần):**
1. Tải `colab_bundle.zip` lên Google Drive, vào thư mục **`MyDrive/ALL/`**.
2. Menu **Runtime → Change runtime type → T4 GPU**.

Chạy lần lượt từng ô. Kết quả lưu thẳng vào `MyDrive/ALL/results/`. Nếu Colab bị ngắt,
mở lại notebook, chạy ô 1–3 rồi chạy lại ô đang dở: các lần chạy đã xong sẽ được bỏ qua.

| Mã | Mô hình / thí nghiệm |
|---|---|
| E0 | Chỉ dùng màu (hồi quy logistic) — mốc kiểm tra lối tắt màu |
| E1 | CNN tự xây dựng (huấn luyện từ đầu) |
| E2 | ResNet50 (pretrained) |
| E3 | EfficientNet-B0 (pretrained) |
| E4 | MobileNetV3-Large (pretrained) |
| C1–C3 | ResNet50: ảnh xám / nhiễu màu mạnh / chuẩn hóa màu Reinhard |

In [ ]:
# 1. Kiểm tra GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Kết nối Google Drive, giải nén dự án vào ổ cục bộ của Colab
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/ALL'
RESULTS = f'{DRIVE_DIR}/results'
!mkdir -p "{RESULTS}"
!rm -rf /content/All-Classification && unzip -q "{DRIVE_DIR}/colab_bundle.zip" -d /content/All-Classification
%cd /content/All-Classification
!pip install -q timm grad-cam pyyaml

In [ ]:
# 3. KIỂM THỬ (~2 phút): chạy thử toàn bộ code trên dữ liệu giả, kể cả huấn luyện.
#    Phải thấy "OK" ở dòng cuối. Nếu có lỗi: DỪNG LẠI và gửi lỗi cho Claude.
!python -m unittest discover -s tests 2>&1 | tail -15

## E0 — Baseline chỉ dùng màu (CPU, ~1 phút)

In [ ]:
# 4.
!python scripts/color_baseline.py --out "{RESULTS}/color_baseline" | tail -12

## E1–E4 — So sánh 4 mô hình (~45–60 phút trên T4)

In [ ]:
# 5.
!python experiments/run_all.py --out "{RESULTS}/runs" --skip-done --color-baseline "{RESULTS}/color_baseline" --group model

## C1–C3 — Kiểm tra lối tắt màu với ResNet50 (~35 phút)

In [ ]:
# 6.
!python experiments/run_all.py --out "{RESULTS}/runs" --skip-done --color-baseline "{RESULTS}/color_baseline" --group color

## (Tùy chọn) Lặp lại 4 mô hình với 3 seed
Báo cáo trung bình ± độ lệch chuẩn thay vì một lần chạy — kết luận so sánh chắc chắn hơn.
Tốn thêm ~1,5–2 giờ; có thể chạy qua nhiều phiên Colab (phần đã xong được bỏ qua).

In [ ]:
# 7. (tùy chọn)
!python experiments/run_all.py --out "{RESULTS}/runs" --skip-done --color-baseline "{RESULTS}/color_baseline" --group model --seeds 42 43 44

## Kết quả so sánh

In [ ]:
# 8. Tạo lại báo cáo và hiển thị
!python scripts/compare_models.py --runs "{RESULTS}/runs" --color-baseline "{RESULTS}/color_baseline" > /dev/null
from IPython.display import Markdown, Image, display
from pathlib import Path
R = Path(RESULTS) / 'runs'
display(Markdown((R / 'COMPARISON.md').read_text().split('## Hình')[0]))
for f in ['fig_accuracy_f1.png', 'fig_roc_malignant.png', 'fig_accuracy_vs_params.png']:
    if (R / f).exists():
        display(Image(str(R / f), width=620))

In [ ]:
# 9. Ma trận nhầm lẫn từng mô hình
for p in sorted(R.glob('*_s42_*/confusion_test.png')):
    print(p.parent.name); display(Image(str(p), width=360))

## Bước 5 — Giải thích mô hình: Grad-CAM, ảnh sai, mô hình nhìn vào đâu (~5 phút)
Mỗi lần chạy có thêm thư mục `explain/` (hình + `summary.md`); bảng tổng hợp ở `runs/EXPLAIN.md`.

In [ ]:
# 10.
!python scripts/explain.py --runs "{RESULTS}/runs" | grep -v '^$'
display(Markdown((R / 'EXPLAIN.md').read_text().split('![](')[0]))
display(Image(str(R / 'explain_compare_gradcam.png'), width=900))

In [ ]:
# 11. Chi tiết một mô hình (đổi tên để xem mô hình khác)
E = sorted(R.glob('resnet50_rgb_s42_*'))[-1] / 'explain'
display(Markdown((E / 'summary.md').read_text()))
for f in ['mask_check.png', 'cells_erased_check.png', 'correct_gradcam.png'] + sorted(p.name for p in E.glob('errors_gradcam_*.png')):
    print(f); display(Image(str(E / f), width=760))